<a href="https://colab.research.google.com/github/Joel-Wang-Math/MIDAS-AI-for-Engineers/blob/main/Copy_of_day02_loading_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loading Data into Pandas

Before you can analyze data, you need to **load** it. Data comes from many sources — URLs, files on your computer, built-in library datasets, or even scattered across many files in a folder.

This short notebook shows four common patterns for getting data into a pandas DataFrame.

In [1]:
import pandas as pd

---
## 1. From a URL

If a CSV file is hosted online, you can pass the URL directly to `pd.read_csv()`. This is the easiest way to load publicly available datasets — no downloads needed.

In [2]:
url = "https://raw.githubusercontent.com/plotly/datasets/master/gapminder_unfiltered.csv"
df_from_url = pd.read_csv(url)

print(f"Loaded {df_from_url.shape[0]} rows, {df_from_url.shape[1]} columns")
df_from_url.head()

Loaded 3313 rows, 6 columns


,country,continent,year,lifeExp,pop,gdpPercap
0,Afghanistan,Asia,1952,28.801,8425333,779.445314
1,Afghanistan,Asia,1957,30.332,9240934,820.853030
2,Afghanistan,Asia,1962,31.997,10267083,853.100710
3,Afghanistan,Asia,1967,34.020,11537966,836.197138
4,Afghanistan,Asia,1972,36.088,13079460,739.981106


This works for any public URL that points to a CSV file. Many scientific datasets on GitHub, government data portals, and data repositories can be loaded this way.

---
## 2. From a library

Many Python libraries ship with **built-in example datasets** for practice and testing. This is convenient because there's nothing to download or configure.

In [3]:
import seaborn as sns

penguins = sns.load_dataset("penguins")

print(f"Loaded {penguins.shape[0]} rows, {penguins.shape[1]} columns")
penguins.head()

Loaded 344 rows, 7 columns


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


The [Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/) dataset contains measurements of three penguin species from Antarctica — a fun alternative to the classic Iris dataset.

Other libraries with built-in datasets include `sklearn.datasets`, `statsmodels.datasets`, and `plotly.data`.

---
## 3. From a local file

The most common real-world pattern: reading a CSV file that lives on your computer (or in your project folder).

*In real projects the file is already sitting on your disk. To keep this notebook self-contained, the next cell **creates** it first, from the URL data we already loaded — which also shows you the flip side of `pd.read_csv()`: writing files with **`.to_csv()`**.*


In [8]:
import os

# A small sample: 10 countries spanning all five continents
countries = ["United States", "Canada", "Brazil", "China", "Japan",
             "Germany", "France", "Nigeria", "Egypt", "Australia"]
sample = df_from_url[df_from_url["country"].isin(countries)]

os.makedirs("data", exist_ok=True)
sample.to_csv("data/gapminder_sample.csv", index=False)
print(f"Wrote data/gapminder_sample.csv ({len(sample)} rows)")

Wrote data/gapminder_sample.csv (383 rows)


In [ ]:
df_from_file = pd.read_csv("data/gapminder_sample.csv")

print(f"Loaded {df_from_file.shape[0]} rows, {df_from_file.shape[1]} columns")
print(f"Countries: {df_from_file['country'].unique().tolist()}")
df_from_file.head()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

The path `"data/gapminder_sample.csv"` is a **relative path** — it means "look for a folder called `data` in the same directory as this notebook, then find `gapminder_sample.csv` inside it."

You can also use an **absolute path** like `"/Users/yourname/Documents/data.csv"`, but relative paths are better for sharing code with others.

Pandas can also read other formats:

| Format | Function |
|---|---|
| CSV | `pd.read_csv()` |
| Excel | `pd.read_excel()` |
| JSON | `pd.read_json()` |
| Parquet | `pd.read_parquet()` |
| HTML tables | `pd.read_html()` |

---
## 4. From multiple files in a directory

Sometimes your data is split across many files — for example, one file per experiment, per year, or per sensor. You need to load them all and combine them into a single DataFrame.

Let's simulate this. First, we'll create a few small CSV files (one per continent), then read them all back in.

### Setup: creating the example files

In [ ]:
import os

# Create a directory for our split files
os.makedirs("data/by_continent", exist_ok=True)

# Split the sample data by continent and save each as a separate CSV
for continent, group in df_from_file.groupby("continent"):
    filepath = f"data/by_continent/{continent}.csv"
    group.to_csv(filepath, index=False)
    print(f"Wrote {filepath} ({len(group)} rows)")

### Finding the files with `glob`

The `glob` module finds files matching a pattern. The `*` wildcard matches any filename.

**🤔 Guess first:** The next cell asks `glob` for every CSV in `data/by_continent/`. Before running it, predict **how many files** it will find. (Hint: the split was one file per continent — how many continents are in our 10-country sample?)


In [ ]:
import glob

# Find all CSV files in the by_continent folder
csv_files = sorted(glob.glob("data/by_continent/*.csv"))
print(f"Found {len(csv_files)} files:")
for f in csv_files:
    print(f"  {f}")

### Reading and combining with `pd.concat()`

Loop over the files, read each one, and combine them all into a single DataFrame.

In [ ]:
# Read each file into a list of DataFrames
dataframes = []
for filepath in csv_files:
    df_chunk = pd.read_csv(filepath)
    dataframes.append(df_chunk)

# Combine them all into one DataFrame
df_combined = pd.concat(dataframes, ignore_index=True)

print(f"Combined: {df_combined.shape[0]} rows from {len(csv_files)} files")
df_combined.head()

Let's verify we got the same data back:

**🤔 Guess first:** We split one file into several and stacked them back together. Before running the next cell, predict: will the combined DataFrame have the **same shape** as the original, or a different one?


In [ ]:
print(f"Original: {df_from_file.shape}")
print(f"Combined: {df_combined.shape}")
print(f"Same number of rows? {df_from_file.shape[0] == df_combined.shape[0]}")

This pattern — `glob` + loop + `pd.concat()` — is extremely common in scientific work where instruments or simulations produce one file per run.

---
## Exercises

### Exercise 1: Load a different library dataset

Seaborn has several built-in datasets. Load the `"tips"` dataset using `sns.load_dataset("tips")` and display its first 5 rows. How many rows and columns does it have?

In [ ]:
# Your code here


### Exercise 2: Combine files with a twist

Using the files in `data/by_continent/`, read and combine **only** the files for the Americas and Asia into a single DataFrame. How many rows does the result have?

*Hint: you don't need `glob` for this — just read the two files by name (remember that file names are case-sensitive!) and use `pd.concat()`.*

In [ ]:
# Your code here


---
## 🏠 Optional Homework (after hours)

Optional exercises for tonight — they only use what's in this notebook. Bring questions tomorrow!

### Homework 1: Meet the Titanic

Load seaborn's built-in `"titanic"` dataset. How many rows and columns does it have? Use `.isna().sum()` to see which columns have missing values — you'll be cleaning data like this tomorrow!


In [ ]:
# Homework 1 — your code here


### Homework 2: Split by year

Mirror what we did in class, with a twist: split the sample data **by year** instead of by continent (one CSV per year in a `data/by_year/` folder), then use `glob` + a loop + `pd.concat()` to put it back together. Verify the shape matches.

*Hint: Because Colab environments start fresh every time, you will need to create the new folder before saving files into it! Use `os.makedirs("data/by_year", exist_ok=True)` first. Then, use `glob` + a loop + `pd.concat()` to pull everything back together.*


In [ ]:
# Homework 2 — your code here


### Homework 3: Round trip

Take your combined Americas + Asia DataFrame from Exercise 2, save it with `.to_csv("data/americas_asia.csv", index=False)`, then load it back with `pd.read_csv()`. Check the reloaded copy against the original — same shape? Try `df1.equals(df2)` for a stricter test.


In [ ]:
# Homework 3 — your code here


---
## What's next?

Now that you can get data *into* pandas from anywhere, the real work begins: **Data Cleaning** — finding and fixing the missing values, wrong data types, duplicates, and outliers hiding in real-world datasets.


---
## Key Points

| Source | How to load |
|---|---|
| URL | `pd.read_csv("https://...")` |
| Library dataset | e.g., `sns.load_dataset("penguins")` |
| Local file | `pd.read_csv("path/to/file.csv")` |
| Multiple files | `glob.glob()` + loop + `pd.concat()` |

- Use **relative paths** (`"data/file.csv"`) over absolute paths for portability.
- Pandas supports many formats beyond CSV: Excel, JSON, Parquet, and more.
- When combining files, `pd.concat(list_of_dataframes, ignore_index=True)` stacks them row-wise.